# MEDAL: one prompt, MCTS initialization, then decoding

[Open in Colab](https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/MEDAL_MCTS_Demo.ipynb)

Run this notebook on its own. It loads **one LLaDA model**, shows the shared E0 decomposition prompt, runs the **MEDAL no-search control**, inspects up to **three initialization candidates** or the selected fallback, and completes the selected partial state. Live counters show search NFEs against the budget, evaluated depth, and candidate count. Reports include E0's MEDAL metrics and the existing stage costs.

Decomposition uses the fixed pens/muffins examples **inside the prompt**, with no separate planning forward. The optional GSM8K batch at the end is disabled by default and reuses the same loaded model.

## Setup — run once

Use a fresh L4/A100 Colab runtime and the LLaDA dependency profile. This demo keeps weights in the notebook kernel so you can rerun decoding without reloading the model. Setup and checks are kept here; the action cells below stay short. The private-repository checkout uses the same Colab secret/hidden-token flow as the other notebooks.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local checkout: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
                str(REPO_DIR)], check=True)
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM") if IN_COLAB else REPO_DIR / ".cache/medal"
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
DEMONSTRATIONS_PATH = DRIVE_ROOT / "manifests/gsm8k_decomposition_train.json"

In [ ]:
import json
import torch
from dlm_project.models.profiles import get_profile
from dlm_project.experiments.config import load_config
from dlm_project.tasks.gsm8k.e0_tokens import resolve_termination_tokens, decode_output
from dlm_project.decoding.medal_no_search import decode as decode_medal_no_search
from dlm_project.decoding.decoder_result import CountingModel
from dlm_project.decoding.medal_demo import (
    DEFAULT_QUESTION, MCTSConfig, load_demo_model, prepare_demo_prompt, show_prompt,
    run_initialization, show_candidates, show_metrics, run_completion, run_demo_batch,
    show_no_search_metrics,
)
from dlm_project.tasks.gsm8k.evaluation import score_answer
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before loading LLaDA.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Use a GPU with bfloat16 support, such as L4 or A100.")
print("GPU:", torch.cuda.get_device_name(0))

### Model source and prompt settings

The source chooser first looks for the runtime copy, then the existing Drive snapshot, then uses the pinned Hub checkpoint. Set `MODEL_PATH` to another stored snapshot if needed. An existing Drive snapshot does not need a new download. The shared E0 prompt uses fixed examples and does not download training demonstrations.

Change `QUESTION`, or set `PROMPT_FILE` to a UTF-8 file containing your question. Shared defaults come from `configs/gsm8k/experiment_0.yaml`: 256 output positions, initialization depth 5, C=3, K1=3, K2=5, gamma 10, exploration 0.01 and seed 1. `SEARCH_BUDGET=512` caps actual search forwards; use 1024 or 2048 to explore larger budgets. Set `INIT_TOKENS=7` for an explicitly extended run. Search runs until the cap or an exhausted frontier, and C only limits the displayed shortlist.

Set `REFERENCE_SOLUTION` to a marked answer such as `"#### 42"` for a custom question. The unchanged default bookstore question is scored against `"#### 27"`; other questions have unavailable accuracy until you supply a reference.

In [ ]:
model_name = get_profile("llada")["repo_id"].split("/")[-1]
stored_models = [Path("/content/models") / model_name, DRIVE_ROOT / "models" / model_name]
MODEL_PATH = next((str(path) for path in stored_models if (path / "config.json").is_file()),
                  get_profile("llada")["repo_id"])
QUESTION = DEFAULT_QUESTION
PROMPT_FILE = None
REFERENCE_SOLUTION = None  # Optional reference for a custom question.
E0_SETTINGS = load_config(REPO_DIR / "configs/gsm8k/experiment_0.yaml")
SEARCH_BUDGET = 512  # Alternatives: 1024, 2048. Each run starts a fresh tree.
INIT_TOKENS = E0_SETTINGS["decoding"]["mcts"]["init_tokens"]  # 5; extended runs may use 7.
GEN_LENGTH, SEED, TEMPERATURE = E0_SETTINGS["generation"]["gen_length"], E0_SETTINGS["decoding"]["seed"], 1.0
PROMPT_SETTINGS = E0_SETTINGS["prompt"]
MCTS_SETTINGS = MCTSConfig(**{**E0_SETTINGS["decoding"]["mcts"], "init_tokens": INIT_TOKENS,
                            "stop_mode": "budget", "max_model_calls": SEARCH_BUDGET, "max_simulations": None})
MCTS_SETTINGS.validate_length(GEN_LENGTH)
print("Model source:", MODEL_PATH)
print(f"Search cap: {SEARCH_BUDGET} actual forwards | target depth: {INIT_TOKENS}")

## 1. Load the model

Run once. Initialization and every later completion share this model.

In [ ]:
llada = load_demo_model(model_path=MODEL_PATH, device="cuda",
                        local_files_only=Path(MODEL_PATH).is_dir())
model, tokenizer = llada.model, llada.tokenizer
termination = resolve_termination_tokens(tokenizer, model, llada.mask_id)
TERMINATION_IDS = termination["token_ids"]

## 2. Load the prompt and show its decomposition

The full E0 two-example prompt is available as `prompt`; `decomposition` contains its metadata and timing. The prompt ends at `Problem: <question>`, without a trailing `1.`. Preparation includes rendering, chat formatting, tokenization, and input transfer. Fixed examples require no demonstration/cache bootstrap and no model forward generates a separate plan.

In [ ]:
question = Path(PROMPT_FILE).read_text(encoding="utf-8") if PROMPT_FILE else QUESTION
prompt, prompt_ids, decomposition = prepare_demo_prompt(
    llada, question, DEMONSTRATIONS_PATH, prompt_settings=PROMPT_SETTINGS)
show_prompt(question, prompt=prompt)

### 2.1. MEDAL completion without search

Run the exact MEDAL downstream decoder from an entirely masked output before running MCTS. This control uses the same loaded model, prepared prompt, generation length, temperature and completion-seed derivation as searched MEDAL. It performs zero search forwards and commits one output token per forward: 256 NFEs with the default output length.

EOS/EOT are permitted during completion; all positions are filled and scoring uses the prefix before the first output terminator. Full output IDs and raw text remain in `no_search_output`. The result and metrics stay separate from `initialization`, `completion`, and the later search report. Rerunning this block performs only the control, and its cost is not added to searched MEDAL's totals.

In [ ]:
no_search_model = CountingModel(model)
no_search = decode_medal_no_search(no_search_model, prompt_ids.clone(), llada.mask_id,
    gen_length=GEN_LENGTH, steps=GEN_LENGTH, block_length=GEN_LENGTH, temperature=TEMPERATURE,
    seed=SEED, mcts=MCTS_SETTINGS, tokenizer=tokenizer)
assert no_search.total_nfes == no_search_model.calls == GEN_LENGTH

In [ ]:
no_search_output = decode_output(tokenizer, no_search.token_ids[0, prompt_ids.shape[1]:].tolist(), TERMINATION_IDS)
print("MEDAL completion without search")
print("Scored prefix:\n" + no_search_output["scoring_text"] + "\n\nFull output:\n" + no_search_output["raw_output"])

In [ ]:
no_search_reference = REFERENCE_SOLUTION
if no_search_reference is None and question.strip() == DEFAULT_QUESTION: no_search_reference = "#### 27"
no_search_score = score_answer(no_search_output["scoring_text"], no_search_reference)
no_search_metrics = show_no_search_metrics(no_search, score=no_search_score,
    prompt_seconds=decomposition["prompt_wall_seconds"], setup_seconds=decomposition["setup_wall_seconds"])

## 3. Initialize a partial state with the current MCTS

This cell performs search only. The root and every evaluated child count against `SEARCH_BUDGET`. Progress refreshes after the root, every 32 forwards, depth increases, and termination. Depth counts fully evaluated states. Search continues after three candidates; it stops at the cap or frontier exhaustion. EOS/EOT are excluded before search top-K selection; native probabilities, entropy, margins and IG are unchanged.

Keep `initialization` to reuse its selected partial state when changing the later sampler. Progress printing is timed separately from search.

In [ ]:
initialization = run_initialization(
    model, prompt_ids, llada.mask_id, gen_length=GEN_LENGTH,
    mcts=MCTS_SETTINGS, seed=SEED, tokenizer=tokenizer,
    search_excluded_token_ids=TERMINATION_IDS, show_progress=True)

### 3.1. Top three C candidates

Inspect up to three distinct, fully evaluated target-depth states ranked by cached mean tree reward, breaking ties by earlier node ID. Only the highest-ranked state is completed. `Q` is the cached selection value; at target-depth leaves it is the **last parent-to-child normalized entropy reduction**. Root-to-candidate entropy change is shown separately and does not change selection. These displays use cached statistics and add no NFE.

One or two candidates form an underfilled shortlist without requiring fallback. With no target-depth candidate, the display shows the selected deepest evaluated state, breaking ties by reward and node ID; if no child was evaluated it shows the root. Budget exhaustion still produces an answer.

In [ ]:
show_candidates(initialization, tokenizer=tokenizer)

### 3.2. Initialization cost

NFE counts actual model forwards. Candidate generation includes one root forward and filtering from cached child logits; each evaluated child adds one forward for its entropy and reward. Selection, child creation, backpropagation, and cached ranking add no forwards. Initialization performs **no full rollout**.

Stage times are exclusive: filtering time is removed from child evaluation before reporting it under candidate generation. Bookkeeping includes trace construction and timing overhead. CUDA boundaries are synchronized, which adds profiling overhead. Combined decoding time is search + this completion; pipeline time additionally includes prompt preparation. Model loading, demonstration/cache setup, live progress printing, scoring, output display, and artifact export are separate.

Reports include search/completion/total NFEs and time, budget utilization, selected/maximum/average evaluated depth, candidates, shortlist size and underfill, selection source, fallback and termination reason. Expanded parents are nodes that produced evaluated children; average branching factor is evaluated children / expanded parents. Unique states count the root and deduplicate equivalent states, while evaluated children count repeated evaluations too. Model evaluations equal NFE because batch size is one.

In [ ]:
metrics = show_metrics(initialization,
                       prompt_seconds=decomposition["prompt_wall_seconds"],
                       setup_seconds=decomposition["setup_wall_seconds"])

## 4. Complete the selected state

`"medal"` uses the unchanged adjusted-score softmax completion at T=1. Copy this cell and change only `DECODING_METHOD` when adding a sampler. Every call receives a fresh copy of the same selected partial state; it preserves the prompt and MCTS commitments. Rerunning this cell performs completion only.

All completion positions are filled and EOS/EOT are permitted. The scored text is the prefix before the first termination token in output order. `completion.decoded_output` retains full/scored token IDs, raw text and the termination boundary for inspection.

In [ ]:
DECODING_METHOD = "medal"  # Later: "my_sampler:decode" or an imported callable.
completion = run_completion(initialization, method=DECODING_METHOD, seed=SEED, temperature=TEMPERATURE,
                            tokenizer=tokenizer, termination_token_ids=TERMINATION_IDS)
response = completion.decoded_output["scoring_text"]
print("Scored prefix:\n" + response + "\n\nFull output:\n" + completion.decoded_output["raw_output"])

In [ ]:
reference = REFERENCE_SOLUTION
if reference is None and question.strip() == DEFAULT_QUESTION:
    reference = "#### 27"
score = score_answer(response, reference)

In [ ]:
metrics = show_metrics(initialization, completion,
                       prompt_seconds=decomposition["prompt_wall_seconds"],
                       setup_seconds=decomposition["setup_wall_seconds"], score=score)

### Adding your sampler later

Put `my_sampler.py` in `REPO_DIR` (or another importable directory) and set `DECODING_METHOD = "my_sampler:decode"`. You can also pass an imported function directly. Its interface is:

```python
def decode(model, state, prompt_len, mask_id, config, *, seed, temperature, tokenizer):
    ...
```

`state` is a CPU `torch.long` tensor shaped `[1, prompt_length + output_length]`. Move inference inputs to `model.device` as needed, fill its remaining output masks, and return an object with `token_ids` of the same shape (`CompletionResult` is already available in `dlm_project.decoding.mcts_decoding`). A `trace` list is optional. Use the supplied model so every forward is counted. The demo verifies that the prompt and initialized tokens survive; checks stay inside the helper.

Each completion report includes the initialization cost **once** plus that completion's actual forwards and time. Earlier sampler trials are not added. `metrics` is a JSON-ready dictionary. Accuracy uses the marked answer in the prefix before the first EOS/EOT after generation; missing or malformed generated answers are incorrect when a reference exists. An unavailable reference is not counted as an incorrect answer. Parse success, correctness and a Wilson 95% accuracy interval are reported for a supplied reference, labeled n=1. Zero remaining masks alone does not establish correctness.

The legacy CLI retains its original prompt and quota defaults; its separate entry point is:

```bash
python scripts/demo_medal.py --model-path /path/to/LLaDA-8B-Instruct --local-files-only \
    --prompt-file /path/to/question.txt
```

## 5. Why can this take a long time?

For the default `"medal"` completion, **total NFE = actual search NFEs + remaining masks at completion**. With B=512 fully consumed and selected depth 5, the total is **512 + (256 - 5) = 763 forwards**. A depth-7 selection needs 249 completion forwards. Frontier exhaustion can use less than B; fallback to a shallower state needs more completion forwards. The cap applies to search, so it is not an end-to-end compute cap.

Each child is evaluated separately on the full prompt-plus-output sequence. Search can spend its budget exploring branches without reaching the requested depth. Finding three candidates does not stop budget mode; completion starts from the best evaluated target-depth state, or an evaluated fallback if none exists.

Each state requires full-vocabulary probabilities, entropy, margins, and action filtering. Final decoding samples over all legal position-token pairs, so reducing tree top-K does not reduce that sampler's action space. CPU tree states transfer to the GPU and compact statistics back to the CPU. CUDA synchronization and detailed token/entropy traces add profiling overhead.

The largest measured stages below show where this run spent time. Change `SEARCH_BUDGET` to 512, 1024 or 2048 in the existing settings cell to explore the trade-off with a fresh tree. Budget use, evaluated depth and fallback metrics show what each cap achieved. Completion reruns reuse the saved initialization.

In [ ]:
print("Largest measured stages:")
for row in sorted(metrics["stages"], key=lambda row: row["wall_seconds"], reverse=True)[:3]:
    print(f"{row['substage']}: {row['wall_seconds']:.3f}s ({row['nfe']} NFE)")

## 6. Optional GSM8K accuracy batch

Leave `RUN_BATCH = False` for the single-prompt demo. Enabling it evaluates the first `BATCH_SIZE` test rows from the existing pinned test manifest, using this loaded model and the same E0 prompt, EOS policy, search budget and completion method. The test manifest is loaded only when enabled. Defaults evaluate 10 questions; each question runs its own initialization and completion.

The helper saves `config.json`, one record per completed question in `predictions.jsonl`, and aggregate costs/accuracy in `summary.json` under a fresh run directory. Records retain raw and scored outputs. Summary metrics include Wilson 95% accuracy intervals, parse/fallback rates, search/completion/total NFEs and runtime, budget utilization, depths, candidate/shortlist counts and termination reasons.

Accuracy is correct / processed, with parse failures counted as incorrect and fallback answers scored normally. An interrupted or failed batch retains completed records and an explicit partial status. Aggregate costs cover completed questions, excluding work on an unfinished question. Model loading is shared; dataset/cache setup and live progress printing are separate from decoding time.

In [ ]:
RUN_BATCH = False
BATCH_SIZE = 10
BATCH_MANIFEST = DRIVE_ROOT / "manifests/gsm8k_test.json"
BATCH_OUTPUT = DRIVE_ROOT / "results/medal_demo"

In [ ]:
if RUN_BATCH:
    batch_directory, batch_summary = run_demo_batch(llada, DEMONSTRATIONS_PATH, BATCH_MANIFEST, BATCH_OUTPUT,
        num_samples=BATCH_SIZE, gen_length=GEN_LENGTH, mcts=MCTS_SETTINGS, seed=SEED, temperature=TEMPERATURE,
        method=DECODING_METHOD, cache_dir=os.environ["HF_DATASETS_CACHE"], prompt_settings=PROMPT_SETTINGS,
        termination_token_ids=TERMINATION_IDS, show_progress=True)

In [ ]:
if RUN_BATCH:
    print(f"Accuracy: {batch_summary['accuracy']:.1%} | processed: {batch_summary['processed']} | total NFE: {batch_summary['total_nfe']}")
    print(f"Mean NFE: {batch_summary['mean_nfe']:.1f} | mean pipeline time: {batch_summary['mean_wall_seconds']:.3f}s")
    print(json.dumps(batch_summary, indent=2))